### Feature Engineering

In [95]:
import pandas as pd

df = pd.read_csv('../data/02_interim/03_clean_races_info.csv', parse_dates=['date', 'dob'])

#### 1. Adding Driver Age at Race Date to Dataset

In [96]:
target_idx = df.columns.get_loc('driverId') + 1

df.insert(target_idx, 'driver_age', (df['date'] - df['dob']).dt.days / 365.25)

#### 2. Adding Driver Momentum (Last 3 Races)

In [97]:
df.insert(target_idx + 1, 'driver_momentum',
          df.groupby('driverId')['positionOrder'].transform(
              lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
          ))

In [98]:
df['driver_momentum'] = df['driver_momentum'].fillna(df['grid'])

df[['driverId', 'raceId', 'positionOrder', 'grid', 'driver_momentum']]

,driverId,raceId,positionOrder,grid,driver_momentum
0,30,53,2,1,1.000000
1,13,53,9,2,2.000000
2,18,53,4,3,3.000000
3,4,53,1,4,4.000000
4,31,53,5,5,5.000000
...,...,...,...,...,...
7885,825,1144,16,14,9.333333
7886,848,1144,11,18,17.333333
7887,855,1144,13,15,12.000000
7888,862,1144,15,17,17.000000


#### 3. Adding Constructor/Team Momentum

In [99]:
# Constructor by race mean
team_avg = df.groupby(['constructorId', 'date'])['positionOrder'].mean().reset_index()

team_avg['constructor_momentum'] = team_avg.groupby('constructorId')['positionOrder'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

df = pd.merge(df, team_avg[['constructorId', 'date', 'constructor_momentum']], on=['constructorId', 'date'], how='left', validate='many_to_many')

In [100]:
df['constructor_momentum'] = df['constructor_momentum'].fillna(
    df.groupby(['constructorId', 'raceId'])['grid'].transform('mean')
)

In [101]:
cols = df.columns.tolist()
col_to_move = cols.pop(cols.index('constructor_momentum'))

target_idx = cols.index('constructorId') + 1

cols.insert(target_idx, col_to_move)

df = df[cols]

#### 4. Adding Driver Skill by Circuit

In [102]:
df['driver_track_affinity'] = df.groupby(['driverId', 'circuitId'])['positionOrder'].transform(
    lambda x: x.shift(1).expanding(min_periods=1).mean()
)

df['driver_track_affinity'] = df['driver_track_affinity'].fillna(df['driver_momentum'])